# 02 — Clean

Selects, parses, and flattens raw data from 01_concat outputs.

## Inputs
- `data/daily_raw.csv` — 887 rows, 51 columns
- `data/activities_raw.csv` — 259 rows, 119 columns

## Key transformations
- Daily: selected 18 core columns, parsed nested `allDayStress`, `bodyBattery`, and `respiration` fields into flat columns
- Activities: selected 34 core columns, converted `duration` from ms to minutes, converted HR zone columns from ms to minutes, dropped `workoutFeel` and `workoutRpe` (>84% null)

## Outputs
- `data/daily_clean.csv` — 887 rows, 30 columns
- `data/activities_clean.csv` — 259 rows, 32 columns

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Paths
project_root = Path().resolve().parent
data_dir = project_root / "data"

df_daily = pd.read_csv(data_dir / "daily_raw.csv")
df_activities = pd.read_csv(data_dir / "activities_raw.csv")

print(f"daily: {df_daily.shape}")
print(f"activities: {df_activities.shape}")

daily: (887, 51)
activities: (259, 119)


In [3]:
# Inspect daily columns
df_daily.columns.tolist()

['userProfilePK',
 'calendarDate',
 'uuid',
 'durationInMilliseconds',
 'totalKilocalories',
 'activeKilocalories',
 'bmrKilocalories',
 'wellnessKilocalories',
 'remainingKilocalories',
 'wellnessTotalKilocalories',
 'wellnessActiveKilocalories',
 'totalSteps',
 'dailyStepGoal',
 'totalDistanceMeters',
 'wellnessDistanceMeters',
 'wellnessStartTimeGmt',
 'wellnessEndTimeGmt',
 'wellnessStartTimeLocal',
 'wellnessEndTimeLocal',
 'highlyActiveSeconds',
 'activeSeconds',
 'moderateIntensityMinutes',
 'vigorousIntensityMinutes',
 'floorsAscendedInMeters',
 'floorsDescendedInMeters',
 'userIntensityMinutesGoal',
 'userFloorsAscendedGoal',
 'minHeartRate',
 'maxHeartRate',
 'restingHeartRate',
 'currentDayRestingHeartRate',
 'restingHeartRateTimestamp',
 'includesWellnessData',
 'includesActivityData',
 'includesCalorieConsumedData',
 'includesSingleMeasurement',
 'includesContinuousMeasurement',
 'includesAllDayPulseOx',
 'includesSleepPulseOx',
 'source',
 'allDayStress',
 'bodyBattery',


In [4]:
# Select useful columns
keep_cols = [
    'calendarDate',
    'totalSteps',
    'totalDistanceMeters',
    'activeSeconds',
    'highlyActiveSeconds',
    'moderateIntensityMinutes',
    'vigorousIntensityMinutes',
    'floorsAscendedInMeters',
    'floorsDescendedInMeters',
    'totalKilocalories',
    'activeKilocalories',
    'bmrKilocalories',
    'minHeartRate',
    'maxHeartRate',
    'restingHeartRate',
    'allDayStress',
    'bodyBattery',
    'respiration',
]

df = df_daily[keep_cols].copy()

# Parse date
df['calendarDate'] = pd.to_datetime(df['calendarDate'])

print(df.shape)
df.head(2)

(887, 18)


,calendarDate,totalSteps,totalDistanceMeters,activeSeconds,highlyActiveSeconds,moderateIntensityMinutes,vigorousIntensityMinutes,floorsAscendedInMeters,floorsDescendedInMeters,totalKilocalories,activeKilocalories,bmrKilocalories,minHeartRate,maxHeartRate,restingHeartRate,allDayStress,bodyBattery,respiration
0,2023-12-10,416.0,331.0,644,36,0,0,7.815,11.794,2028.0,26.0,2002.0,56.0,104.0,63.0,"{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '..."
1,2023-12-11,5464.0,6174.0,5650,354,52,1,47.014,61.370,2435.0,433.0,2002.0,48.0,129.0,57.0,"{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '...","{'userProfilePK': 118214492, 'calendarDate': '..."


In [5]:
df.isnull().sum()

calendarDate                0
totalSteps                  3
totalDistanceMeters         3
activeSeconds               0
highlyActiveSeconds         0
moderateIntensityMinutes    0
vigorousIntensityMinutes    0
floorsAscendedInMeters      0
floorsDescendedInMeters     0
totalKilocalories           0
activeKilocalories          0
bmrKilocalories             0
minHeartRate                8
maxHeartRate                8
restingHeartRate            9
allDayStress                0
bodyBattery                 8
respiration                 8
dtype: int64

In [6]:
import ast

# Check what type these are after CSV loading
print(type(df['allDayStress'].iloc[0]))
print(df['allDayStress'].iloc[0])

<class 'str'>
{'userProfilePK': 118214492, 'calendarDate': '2023-12-10', 'aggregatorList': [{'type': 'TOTAL', 'averageStressLevel': 39, 'averageStressLevelIntensity': 31, 'maxStressLevel': 89, 'stressIntensityCount': 259, 'stressOffWristCount': 57, 'stressTooActiveCount': 22, 'totalStressCount': 338, 'totalStressIntensity': -2224, 'stressDuration': 10560, 'restDuration': 4980, 'activityDuration': 1320, 'uncategorizedDuration': 3420, 'totalDuration': 20280, 'lowDuration': 5520, 'mediumDuration': 4140, 'highDuration': 900}, {'type': 'AWAKE', 'averageStressLevel': 39, 'averageStressLevelIntensity': 31, 'maxStressLevel': 89, 'stressIntensityCount': 259, 'stressOffWristCount': 57, 'stressTooActiveCount': 22, 'totalStressCount': 338, 'totalStressIntensity': -2224, 'stressDuration': 10560, 'restDuration': 4980, 'activityDuration': 1320, 'uncategorizedDuration': 3420, 'totalDuration': 20280, 'lowDuration': 5520, 'mediumDuration': 4140, 'highDuration': 900}, {'type': 'ASLEEP', 'averageStressLev

In [7]:
# Parse allDayStress and extract TOTAL aggregator
def parse_stress(val):
    if pd.isna(val):
        return {}
    d = ast.literal_eval(val)
    for agg in d.get('aggregatorList', []):
        if agg.get('type') == 'TOTAL':
            return {
                'stress_avg': agg.get('averageStressLevel'),
                'stress_max': agg.get('maxStressLevel'),
                'stress_rest_duration': agg.get('restDuration'),
                'stress_low_duration': agg.get('lowDuration'),
                'stress_medium_duration': agg.get('mediumDuration'),
                'stress_high_duration': agg.get('highDuration'),
            }
    return {}

stress_df = df['allDayStress'].apply(parse_stress).apply(pd.Series)
print(stress_df.shape)
stress_df.head(2)

(887, 6)


,stress_avg,stress_max,stress_rest_duration,stress_low_duration,stress_medium_duration,stress_high_duration
0,39.0,89.0,4980.0,5520.0,4140.0,900.0
1,35.0,99.0,29880.0,10260.0,12720.0,9180.0


In [8]:
# Parse bodyBattery
def parse_battery(val):
    if pd.isna(val):
        return {}
    d = ast.literal_eval(val)
    result = {
        'battery_charged': d.get('chargedValue'),
        'battery_drained': d.get('drainedValue'),
    }
    for stat in d.get('bodyBatteryStatList', []):
        t = stat.get('bodyBatteryStatType')
        if t == 'HIGHEST':
            result['battery_high'] = stat.get('statsValue')
        elif t == 'LOWEST':
            result['battery_low'] = stat.get('statsValue')
        elif t == 'STARTOFDAY':
            result['battery_start'] = stat.get('statsValue')
        elif t == 'ENDOFDAY':
            result['battery_end'] = stat.get('statsValue')
    return result

battery_df = df['bodyBattery'].apply(parse_battery).apply(pd.Series)
print(battery_df.shape)
battery_df.head(2)

(887, 6)


,battery_charged,battery_drained,battery_high,battery_low,battery_end,battery_start
0,6.0,11.0,16.0,5.0,11.0,16.0
1,89.0,92.0,94.0,6.0,8.0,11.0


In [9]:
# Parse respiration
def parse_respiration(val):
    if pd.isna(val):
        return {}
    d = ast.literal_eval(val)
    return {
        'respiration_avg_waking': d.get('avgWakingRespirationValue'),
        'respiration_high': d.get('highestRespirationValue'),
        'respiration_low': d.get('lowestRespirationValue'),
    }

respiration_df = df['respiration'].apply(parse_respiration).apply(pd.Series)
print(respiration_df.shape)
respiration_df.head(2)

(887, 3)


,respiration_avg_waking,respiration_high,respiration_low
0,14.0,17.0,10.0
1,13.0,21.0,8.0


In [10]:
# Drop nested columns and join parsed ones
df_daily_clean = df.drop(columns=['allDayStress', 'bodyBattery', 'respiration'])
df_daily_clean = pd.concat([df_daily_clean, stress_df, battery_df, respiration_df], axis=1)

print(df_daily_clean.shape)
df_daily_clean.head(2)

(887, 30)


,calendarDate,totalSteps,totalDistanceMeters,activeSeconds,highlyActiveSeconds,moderateIntensityMinutes,vigorousIntensityMinutes,floorsAscendedInMeters,floorsDescendedInMeters,totalKilocalories,...,stress_high_duration,battery_charged,battery_drained,battery_high,battery_low,battery_end,battery_start,respiration_avg_waking,respiration_high,respiration_low
0,2023-12-10,416.0,331.0,644,36,0,0,7.815,11.794,2028.0,...,900.0,6.0,11.0,16.0,5.0,11.0,16.0,14.0,17.0,10.0
1,2023-12-11,5464.0,6174.0,5650,354,52,1,47.014,61.370,2435.0,...,9180.0,89.0,92.0,94.0,6.0,8.0,11.0,13.0,21.0,8.0


In [11]:
df_activities.columns.tolist()

['activityId',
 'uuidMsb',
 'uuidLsb',
 'name',
 'activityType',
 'userProfileId',
 'timeZoneId',
 'beginTimestamp',
 'eventTypeId',
 'rule',
 'sportType',
 'startTimeGmt',
 'startTimeLocal',
 'duration',
 'avgHr',
 'maxHr',
 'minHr',
 'steps',
 'calories',
 'bmrCalories',
 'avgFractionalCadence',
 'maxFractionalCadence',
 'elapsedDuration',
 'movingDuration',
 'deviceId',
 'summarizedExerciseSets',
 'summarizedDiveInfo',
 'manufacturer',
 'lapCount',
 'waterEstimated',
 'activeSets',
 'totalSets',
 'totalReps',
 'minRespirationRate',
 'maxRespirationRate',
 'avgRespirationRate',
 'trainingEffectLabel',
 'startStress',
 'endStress',
 'differenceStress',
 'aerobicTrainingEffectMessage',
 'anaerobicTrainingEffectMessage',
 'moderateIntensityMinutes',
 'vigorousIntensityMinutes',
 'differenceBodyBattery',
 'avgStress',
 'maxStress',
 'workoutFeel',
 'workoutRpe',
 'hrTimeInZone_0',
 'hrTimeInZone_1',
 'hrTimeInZone_2',
 'hrTimeInZone_3',
 'hrTimeInZone_4',
 'hrTimeInZone_5',
 'hrTimeInZon

In [12]:
# Select useful activity columns
activity_keep_cols = [
    'activityId',
    'name',
    'activityType',
    'sportType',
    'beginTimestamp',
    'duration',
    'distance',
    'avgHr',
    'maxHr',
    'minHr',
    'calories',
    'avgSpeed',
    'elevationGain',
    'elevationLoss',
    'moderateIntensityMinutes',
    'vigorousIntensityMinutes',
    'startStress',
    'endStress',
    'differenceStress',
    'avgStress',
    'maxStress',
    'differenceBodyBattery',
    'trainingEffectLabel',
    'aerobicTrainingEffectMessage',
    'anaerobicTrainingEffectMessage',
    'workoutFeel',
    'workoutRpe',
    'hrTimeInZone_0',
    'hrTimeInZone_1',
    'hrTimeInZone_2',
    'hrTimeInZone_3',
    'hrTimeInZone_4',
    'hrTimeInZone_5',
    'vO2MaxValue',
]

df_act_clean = df_activities[activity_keep_cols].copy()

# Convert timestamp to datetime
df_act_clean['date'] = pd.to_datetime(df_act_clean['beginTimestamp'], unit='ms').dt.date
df_act_clean['date'] = pd.to_datetime(df_act_clean['date'])

# Convert duration from milliseconds to minutes
df_act_clean['duration_minutes'] = (df_act_clean['duration'] / 60000).round(1)

df_act_clean = df_act_clean.drop(columns=['beginTimestamp', 'duration'])

print(df_act_clean.shape)
df_act_clean.head(2)

(259, 34)


,activityId,name,activityType,sportType,distance,avgHr,maxHr,minHr,calories,avgSpeed,...,workoutRpe,hrTimeInZone_0,hrTimeInZone_1,hrTimeInZone_2,hrTimeInZone_3,hrTimeInZone_4,hrTimeInZone_5,vO2MaxValue,date,duration_minutes
0,22941249248,Yoga,yoga,TRAINING,NaN,96.0,139.0,43.0,980.46468,NaN,...,30.0,2247996.0,1101926.0,612997.0,8000.0,0.0,0.0,NaN,2026-05-19,66.2
1,22928860541,Yoga,yoga,TRAINING,NaN,67.0,118.0,40.0,377.10180,NaN,...,10.0,3630941.0,53321.0,1998.0,0.0,0.0,0.0,NaN,2026-05-18,61.4


In [13]:
df_act_clean.isnull().sum()

activityId                          0
name                                0
activityType                        0
sportType                          14
distance                           96
avgHr                               1
maxHr                               1
minHr                              94
calories                            0
avgSpeed                           94
elevationGain                     246
elevationLoss                     246
moderateIntensityMinutes           34
vigorousIntensityMinutes           34
startStress                       176
endStress                         176
differenceStress                  176
avgStress                         175
maxStress                         175
differenceBodyBattery               4
trainingEffectLabel                 1
aerobicTrainingEffectMessage        1
anaerobicTrainingEffectMessage      1
workoutFeel                       218
workoutRpe                        221
hrTimeInZone_0                     62
hrTimeInZone

In [14]:
# Convert HR time in zones from milliseconds to minutes
hr_zone_cols = ['hrTimeInZone_0', 'hrTimeInZone_1', 'hrTimeInZone_2',
                'hrTimeInZone_3', 'hrTimeInZone_4', 'hrTimeInZone_5']

df_act_clean[hr_zone_cols] = (df_act_clean[hr_zone_cols] / 60000).round(1)

df_act_clean[hr_zone_cols].head(3)

,hrTimeInZone_0,hrTimeInZone_1,hrTimeInZone_2,hrTimeInZone_3,hrTimeInZone_4,hrTimeInZone_5
0,37.5,18.4,10.2,0.1,0.0,0.0
1,60.5,0.9,0.0,0.0,0.0,0.0
2,41.4,21.1,4.9,0.0,0.0,0.0


In [15]:
# Drop sparse columns
df_act_clean = df_act_clean.drop(columns=['workoutFeel', 'workoutRpe'])

# Save cleaned DataFrames
df_daily_clean.to_csv(data_dir / "daily_clean.csv", index=False)
df_act_clean.to_csv(data_dir / "activities_clean.csv", index=False)

print("Saved:")
print(f"  daily_clean.csv — {len(df_daily_clean)} rows, {len(df_daily_clean.columns)} columns")
print(f"  activities_clean.csv — {len(df_act_clean)} rows, {len(df_act_clean.columns)} columns")

Saved:
  daily_clean.csv — 887 rows, 30 columns
  activities_clean.csv — 259 rows, 32 columns
